In [ ]:
# Install required packages.
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Helper function for visualization.
%matplotlib inline
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA

# Node Classification with Graph Neural Networks

[Pytorch Geometric official tutorials](https://pytorch-geometric.readthedocs.io/en/2.6.1/get_started/colabs.html#official-examples)

This tutorial will teach you how to apply **Graph Neural Networks (GNNs) to the task of node classification**.
Here, we are given the ground-truth labels of only a small subset of nodes, and want to infer the labels for all the remaining nodes (*transductive learning*).

To demonstrate, we make use of the `Cora` dataset, which is a **citation network** where nodes represent documents.
Each node is described by a 1433-dimensional bag-of-words feature vector.
Two documents are connected if there exists a citation link between them.
The task is to infer the category of each document (7 in total).

This dataset was first introduced by [Yang et al. (2016)](https://arxiv.org/abs/1603.08861) as one of the datasets of the `Planetoid` benchmark suite.
We again can make use [PyTorch Geometric](https://github.com/rusty1s/pytorch_geometric) for an easy access to this dataset via [`torch_geometric.datasets.Planetoid`](https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html#torch_geometric.datasets.Planetoid):

In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures


dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())

print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

In [ ]:
data = dataset[0]  # Get the first graph object.


# Gather some statistics about the graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')

In [ ]:
# Randomly we sample 0.8 of the nodes for training, 0.1 for validation, and 0.1 for testing.
num_nodes = data.num_nodes
num_train = int(num_nodes * 0.8)
num_val = int(num_nodes * 0.1)
num_test = num_nodes - num_train - num_val
perm = torch.randperm(num_nodes)
data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.train_mask[perm[:num_train]] = True
data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.val_mask[perm[num_train:num_train + num_val]] = True
data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask[perm[num_train + num_val:]] = True

In [ ]:
data.train_mask

In [ ]:
### To do it with your own data: x is the feature matrix, edge_index is the edgelist
#data = Data(x=x, edge_index=edge_index, ...)


In [ ]:
# Quick overview
print(data)

In [ ]:
data.x

In [ ]:
# How many words in a document?
(data.x[0,:]>0).sum()

In [ ]:
data.edge_index.t()

In [ ]:
pd.Series(data.y).value_counts()

We compute coordinates for the nodes with NetworkX. In large networks (>1000 nodes) drawing edges can slow down significantly the plot, so we extract the coordinates and plot them with Matplotlib.

In [ ]:
from torch_geometric.utils import to_networkx

G = to_networkx(data, to_undirected=True)

pos= nx.spring_layout(G)
xcoord= [coord[0] for coord in pos.values()]
ycoord= [coord[1] for coord in pos.values()]
plt.scatter(xcoord,ycoord, c=data.y,cmap='Set2')

## Non-graph Neural Network (Multi Layer Perceptron)
Classification using only node features, without using the graph information at all.

In [ ]:
from torch.nn import Linear, Dropout
import torch.nn.functional as F


class MLP(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(12345)
        self.lin1 = Linear(dataset.num_node_features, hidden_channels)
        self.lin2 = Linear(hidden_channels, dataset.num_classes)

    def forward(self, x):
        x = self.lin1(x)
        x = x.relu()
        x = F.dropout(x, p=0.2)
        x = self.lin2(x)
        return x

model = MLP(hidden_channels=16)
print(model)
print('N. of weights:',sum(p.numel() for p in model.parameters()))

In [ ]:
model(data.x)

In [ ]:
from torch import softmax
softmax(model(data.x),1)#.sum(1)

In [ ]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)  # Define optimizer.

def train():
      model.train()
      optimizer.zero_grad()  # Clear gradients.
      out = model(data.x)  # Perform a single forward pass.
      loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
      loss.backward()  # Derive gradients.
      optimizer.step()  # Update parameters based on gradients.
      return loss



for epoch in range(1, 201):
    loss = train()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

### Evaluate on test set

In [ ]:
model.eval()
out = model(data.x)
pred = out.argmax(dim=1)  # Use the class with highest probability.

train_correct = pred[data.train_mask] == data.y[data.train_mask]  # Check against ground-truth labels.
train_acc = int(train_correct.sum()) / int(data.train_mask.sum())  # Derive ratio of correct predictions.
print(f'Training set accuracy:{train_acc:.3f}')

test_correct = pred[data.test_mask] == data.y[data.test_mask]  # Check against ground-truth labels.
test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  # Derive ratio of correct predictions.
print(f'Test set accuracy:{test_acc:.3f}')


## Convolutional GNN

In [ ]:
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GCNConv(data.num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(hidden_channels=16)
print(model)
print('N. of weights:',sum(p.numel() for p in model.parameters()))

In [ ]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)  # Define optimizer.

def train():
      model.train()
      optimizer.zero_grad()  # Clear gradients.
      out = model(data.x, data.edge_index)  # Perform a single forward pass.
      loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
      loss.backward()  # Derive gradients.
      optimizer.step()  # Update parameters based on gradients.
      return loss



for epoch in range(1, 201):
    loss = train()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

### Evaluate on test set

In [ ]:
model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)  # Use the class with highest probability.

train_correct = pred[data.train_mask] == data.y[data.train_mask]  # Check against ground-truth labels.
train_acc = int(train_correct.sum()) / int(data.train_mask.sum())  # Derive ratio of correct predictions.
print(f'Training set accuracy:{train_acc:.3f}')

test_correct = pred[data.test_mask] == data.y[data.test_mask]  # Check against ground-truth labels.
test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  # Derive ratio of correct predictions.
print(f'Test set accuracy:{test_acc:.3f}')


### Visualization of the logits (supervised)

In [ ]:
logits= model(data.x,data.edge_index).detach().cpu().numpy()
z= PCA(2).fit_transform(logits)

plt.figure(figsize=(6,6))
plt.scatter(z[:, 0], z[:, 1], s=20, c=data.y, cmap="Set2")
plt.show()

In [ ]:
# With edges
logits= model(data.x,data.edge_index).detach().cpu().numpy()
z= PCA(2).fit_transform(logits)
supervised_pos={i: (z[i,0],z[i,1]) for i in range(len(data.x))}

plt.figure(figsize=(6,6))
plt.scatter(z[:, 0], z[:, 1], s=20, c=data.y, cmap="Set2")
nx.draw_networkx_edges(G,supervised_pos,alpha=.05)
plt.show()

## Node classification without network features
When only the network is available, without any feature at node level.

Workaround: a one-hot encoding for each node (identity matrix).

In [ ]:
data = dataset[0]
data.x = torch.eye(data.num_nodes)

model = GCN(16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)  # Define optimizer.

for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
      print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')



In [ ]:
model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)  # Use the class with highest probability.

train_correct = pred[data.train_mask] == data.y[data.train_mask]  # Check against ground-truth labels.
train_acc = int(train_correct.sum()) / int(data.train_mask.sum())  # Derive ratio of correct predictions.
print(f'Training set accuracy:{train_acc:.3f}')

test_correct = pred[data.test_mask] == data.y[data.test_mask]  # Check against ground-truth labels.
test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  # Derive ratio of correct predictions.
print(f'Test set accuracy:{test_acc:.3f}')


# Node embeddings
Here we generalize the approach to the case when no label are provided at node level, hence we aim to build *unsupervised* or better *self-supervised* node represenations.

We will build a loss function grounded on *contrastive learning*, which mean to learn by showing the GNN pairs of *positive* and *negative* pairs of nodes. A positive pair is composed by two connected nodes, while a negative pair is composed by two nodes without an edge between them.

The loss function aims to maximize the proximity of connected nodes in the embedding space (positive pairs) and minimize the proximity of negative pairs (random sampling them).


![image info](./figures/pos_neg_loss.png "Contrastive loss")

In [ ]:
data = dataset[0]
data.edge_index.shape

In [ ]:
data.edge_index[:,:10]

In [ ]:
# If this does not raise an error, it means the edge exists
G.edges[633,0]

In [ ]:
from torch_geometric.utils import negative_sampling
negative_sampling(data.edge_index).shape

In [ ]:
negative_sampling(data.edge_index)[:,:10]

In [ ]:
# If this raises a KeyError, it means the edge does not exist
G.edges[1916,2517]

In [ ]:
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_size):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, embedding_size)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = GraphSAGE(data.num_features, 16, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def loss_fn(z, edge_index):
    # Positive pairs
    pos_loss = -torch.log(torch.sigmoid((z[edge_index[0]] * z[edge_index[1]]).sum(dim=1))).mean()

    # Negative sampling
    neg_edge_index = negative_sampling(edge_index, num_nodes=z.size(0))
    neg_loss = -torch.log(1 - torch.sigmoid((z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1))).mean()

    return pos_loss + neg_loss

# Training loop
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    # This time we also encode the node features in the embedding
    embeddings = model(data.x, data.edge_index)
    loss = loss_fn(embeddings, data.edge_index)
    loss.backward()
    optimizer.step()
    if epoch%20==0:
      print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

print("Node embeddings shape:", embeddings.shape)

In [ ]:
# With edges
#z= PCA(2).fit_transform(embeddings.detach().cpu().numpy())
z=embeddings.detach().cpu().numpy()
self_supervised_pos={i: (z[i,0],z[i,1]) for i in range(len(data.x))}

plt.figure(figsize=(6,6))
plt.scatter(z[:, 0], z[:, 1], s=20, c=data.y, cmap="Set2")
nx.draw_networkx_edges(G,self_supervised_pos,alpha=.05)
plt.show()